# Simulation II: Sim-to-Real and Learning from Interaction

This lecture connects three layers of modern robotics practice:
how we **build simulators**, how we **bridge the reality gap**, and how we
**learn control policies from interaction**.  The goal is not to memorize a
catalog of tools, but to understand which simulator or transfer strategy is
appropriate for which robotics problem.

We will keep returning to the same small MuJoCo arm example to make the survey
more concrete, and we will close with a compact comparison of the main
sim-to-real and learning recipes.

## Table of Contents

### Part I. Building Simulators
1. [Why Simulate?](#simulation-landscape)
2. [Robot / Scene Description Formats](#formats)
3. [Simulation Platforms](#platforms)
4. [Physics Engines Under the Hood](#physics-engines)
5. [Rendering and Synthetic Data](#rendering)

### Part II. Bridging the Reality Gap
6. [The Reality Gap](#reality-gap)
7. [Domain Randomization](#domain-randomization)
8. [Domain Adaptation and System Identification](#adaptation-sysid)
9. [Teacher-Student Architectures](#teacher-student)
10. [Digital Twins](#digital-twins)

### Part III. Learning from Interaction
11. [RL Formulation for Robotics](#rl-formulation)
12. [Imitation Learning and DAgger](#imitation-dagger)

Practice demos are interleaved throughout the notebook and reuse the same
small MuJoCo examples where possible.


In [ ]:
%matplotlib widget
import sys
from pathlib import Path

_week_dir = Path.cwd()
if str(_week_dir) not in sys.path:
    sys.path.insert(0, str(_week_dir))

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import mediapy as media


### Notation and Conventions

| Symbol | Meaning |
|--------|---------|
| $s, a, r$ | RL state, action, reward |
| $\pi(a \mid s)$ | policy |
| $\tau = (s_0, a_0, s_1, \dots)$ | trajectory / rollout |
| $\theta$ | physical parameter vector used by the simulator |
| $\hat{\theta}$ | identified nominal parameter estimate |
| $o_{\text{priv}}$ | privileged observation available only in simulation |
| $o_{\text{deploy}}$ | deployable observation available on the real robot |
| $\mathcal{D}$ | dataset of demonstrations or aggregated labels |
| `sim` / `real` | simulator-domain and real-world quantities |

- We distinguish the **simulator model** from the **real system** explicitly:
  `sim` is what the physics engine predicts, `real` is what the hardware does.
- Week 6 mixes two viewpoints:
  state-space control notation for RL / IL, and parameter-identification
  notation for sim-to-real.
- When discussing transfer methods, ask three questions:
  which gap does it reduce, what data does it require, and what assumptions
  make it fail?


---
# 1. Why Simulate? <a id="simulation-landscape"></a>

## Part I. Building Simulators

Simulation is the **backbone of modern robot learning**.  Training directly on hardware is slow, expensive, and dangerous — a single RL experiment may require millions of environment steps.  Simulators give us:

| Benefit | What it means in practice |
|---------|--------------------------|
| **Speed** | Thousands of parallel environments on one GPU; minutes instead of weeks |
| **Safety** | No risk of breaking a \$50 k arm while exploring random torques |
| **Scale** | Billions of transitions for RL; unlimited labeled data for perception |
| **Reproducibility** | Deterministic resets, exact state logging, automated ablations |
| **Control** | Vary any physical parameter at will — gravity, friction, lighting |

The price we pay is the **reality gap**: a policy that works in simulation may fail on the real robot.  The rest of this lecture is about understanding simulators deeply, closing that gap, and learning robust policies.

**Roadmap for today:**

```
Simulators ──► Sim-to-Real ──► Learning from Interaction
 │                 │                 │
 ├─ Formats        ├─ Reality gap    ├─ RL formulation
 ├─ Platforms      ├─ Domain Rand.   ├─ Imitation Learning
 ├─ Physics        ├─ Adaptation     ├─ DAgger
 └─ Rendering      ├─ SysID          └─ Hybrid methods
                   ├─ Teacher-Student
                   └─ Digital Twins
```

> **Key takeaway:** Simulation is useful because it is fast, safe, and controllable.
> But those same simplifications create the reality gap.  The rest of the
> lecture is about deciding which parts of the simulator matter for transfer.


---
# 2. Robot / Scene Description Formats <a id="formats"></a>

Before you can simulate, you need a **machine-readable description** of your robot and its environment.  Several competing formats exist, each with different strengths:

| Format | Full name | Strengths | Limitations | Ecosystem |
|--------|-----------|-----------|-------------|-----------|
| **URDF** | Unified Robot Description Format | Simple, ubiquitous in ROS, wide tool support | No closed loops, limited contact spec, no world/scene, no sensors | ROS, PyBullet, MoveIt, pytorch\_kinematics |
| **MJCF** | MuJoCo XML | Rich actuator/contact/sensor models, defaults system, composable | MuJoCo-specific, no world-level scene composition (vs SDF) | MuJoCo, MJX, Playground, dm\_control |
| **SDF** | Simulation Description Format | Worlds + models + lights + sensors, richer than URDF | More verbose, mainly Gazebo ecosystem | Gazebo, Ignition |
| **USD** | Universal Scene Description | Layered scene graphs, non-destructive composition, VFX-grade | Complex tooling, heavier runtime | Isaac Sim, Omniverse, film/VFX pipelines |

### URDF recap (from Week 1)

You already know URDF: a tree of `<link>` (rigid bodies with inertial/visual/collision) connected by `<joint>`.  Key limitation: **tree topology only** — no closed kinematic chains.

### MJCF highlights

MJCF is the native format for MuJoCo.  Compared to URDF it adds:

- **`<default>` classes** — hierarchical parameter inheritance (set friction once, all child geoms inherit).
- **Actuator models** — motors, position/velocity servos, tendons, muscles — decoupled from joints.
- **Contact parameters** — per-geom friction, condim (contact dimensionality), solref/solimp (solver tuning).
- **Sensors** — joint encoders, accelerometers, touch, cameras — first-class XML elements.

Let's load and inspect an MJCF model:


In [ ]:
from lib.mujoco_utils import load_model, make_data, render_frame, print_model_summary

model = load_model("assets/demo_arm.mjcf")
data = make_data(model)

print("=== MJCF model summary ===")
print_model_summary(model)

print("\n=== Contact parameters (first 4 geoms) ===")
for i in range(min(4, model.ngeom)):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, i) or f"geom_{i}"
    print(f"  {name:16s}  friction={model.geom_friction[i]}  "
          f"solref={model.geom_solref[i]}  solimp={model.geom_solimp[i][:3]}")

print("\n=== Rendered frame ===")
frame = render_frame(model, data, width=480, height=360)
media.show_image(frame)


---
# 3. Simulation Platforms <a id="platforms"></a>

A "simulator" bundles a **physics engine**, a **renderer**, and a **scripting API**.  Choosing the right platform depends on your task, hardware, and ecosystem needs.

| Platform | Physics engine | Rendering | GPU-parallel | Key strength | Typical use |
|----------|---------------|-----------|:------------:|-------------|-------------|
| **Gazebo** (Classic / Ignition) | ODE, Bullet, DART, Simbody | Ogre / Ogre2 | ✗ | Deep ROS integration, multi-robot worlds | ROS navigation, manipulation pipelines |
| **PyBullet** | Bullet | OpenGL (built-in) | ✗ | Simple Python API, quick prototyping | RL benchmarks, research baselines |
| **MuJoCo** | MuJoCo engine | OpenGL / EGL | ✗ (single) | Fast & accurate contacts, rich MJCF | Continuous control, contact-rich tasks |
| **MJX** | MuJoCo via JAX | — (headless) | ✓ (JAX) | Batched differentiable sim on GPU/TPU | Massively parallel RL, gradient-based optimisation |
| **MuJoCo Playground** | MJX | Browser (three.js) | ✓ (JAX) | Pre-built tasks + training infra | Quick RL experiments, locomotion / manipulation |
| **Isaac Sim / Lab** | PhysX 5 | RTX ray-tracing | ✓ (GPU) | Photorealistic + GPU-parallel rigid/soft | Synthetic data, large-scale sim-to-real |
| **SAPIEN** | PhysX | Vulkan | Partial | Articulated-object datasets (PartNet) | Manipulation of household objects |
| **Genesis** | Custom multi-physics | Rasterisation | ✓ (GPU) | Multi-physics (rigid, soft, fluid, cloth) | Emerging; large-scale batched sim |

### Quick comparison axes

- **Throughput**: MJX / Isaac Lab >> MuJoCo >> PyBullet >> Gazebo.  GPU-parallel sims can reach $10^6$ steps/sec.
- **Contact fidelity**: MuJoCo ≈ PhysX 5 >> Bullet > ODE for contact-rich manipulation.
- **Photorealism**: Isaac Sim (RTX) >> Gazebo/PyBullet.  Matters for vision-based policies.
- **Ecosystem**: Gazebo ↔ ROS;  MuJoCo ↔ dm\_control / Gymnasium;  Isaac ↔ Omniverse / USD.

> **Key takeaway:** There is no single best simulator.  For RL research MuJoCo/MJX is the default; for production sim-to-real with vision, Isaac Sim leads; for ROS integration, Gazebo remains standard.


---
# 4. Physics Engines Under the Hood <a id="physics-engines"></a>

All physics engines solve the same core problem: advance the state of rigid (and sometimes soft) bodies by one timestep, resolving contacts and constraints.  They differ in **how** they model contacts.

## Contact models

| Engine | Contact model | Key idea | Pros | Cons |
|--------|--------------|----------|------|------|
| **Bullet** | LCP (Linear Complementarity) | Contacts as hard complementarity constraints solved via Lemke / PGS | Physically principled, no penetration | Can be slow / jittery with many contacts |
| **MuJoCo** | Soft complementarity | Contacts as *soft* constraints with tunable stiffness/damping (solref/solimp) | Smooth, fast, differentiable-friendly | Not strictly non-penetrating; needs parameter tuning |
| **PhysX 5** | TGS / PBD hybrid | Temporal Gauss-Seidel + position-based corrections, GPU-batched | Very fast on GPU, handles large scenes | Less accurate for high-precision contacts |

### MuJoCo's contact model in more detail

MuJoCo uses a **convex optimisation** formulation rather than LCP:

$$\min_{\mathbf{f}} \; \frac{1}{2} \mathbf{f}^T A \mathbf{f} + \mathbf{f}^T \mathbf{b}
\quad \text{s.t.} \quad \mathbf{f} \in \mathcal{C}$$

where $\mathbf{f}$ are contact forces, $A$ is related to the inverse inertia, $\mathbf{b}$ encodes velocity-level constraints, and $\mathcal{C}$ is the friction cone.  The softness parameters `solref` (time constant, damping ratio) and `solimp` (impedance) control how contacts behave.

This means MuJoCo contacts are:
- **Smooth** — contact forces change continuously rather than switching abruptly.
- **Tunable** — `solref` and `solimp` let us trade off stiffness and damping.
- **Efficient in practice** — especially for the medium-size articulated systems that dominate robotics benchmarks.

## Beyond rigid bodies

| Phenomenon | Bullet | MuJoCo | PhysX 5 |
|-----------|--------|--------|---------|
| Deformables / soft bodies | ✓ (FEM deformables) | Limited (approximate via tendons / composites) | ✓ (FEM, cloth) |
| Cloth | Limited | ✗ | ✓ (GPU cloth solver) |
| Fluids / particles | SPH (experimental) | ✗ | ✓ (PBD particles, Flex) |
| Cables / tendons | ✗ | ✓ (spatial tendons) | Limited |

Friction and compliance affect **different** things, so we should not teach them with the same setup:

- **Friction** matters when there is tangential sliding at a contact.
- **Compliance / softness** matters when bodies hit and deform the contact constraint.

Let's look at one example of each:


In [ ]:
# Tangential contact: same initial push, different friction
SLIDE_XML = """
<mujoco model="slide_demo">
  <option timestep="0.002" gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 0.1" rgba="0.85 0.85 0.88 1"
          friction="{friction}"/>
    <body name="block" pos="0 0 0.08">
      <freejoint/>
      <geom type="box" size="0.08 0.05 0.05" mass="0.7" rgba="{color}"
            friction="{friction}"/>
    </body>
  </worldbody>
</mujoco>
"""

# Normal direction: same drop, different contact softness
DROP_XML = """
<mujoco model="drop_demo">
  <option timestep="0.001" gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 0.1" rgba="0.85 0.85 0.88 1"
          solref="{solref}" friction="1.0 0.005 0.0001"/>
    <body name="ball" pos="0 0 1.0">
      <freejoint/>
      <geom type="sphere" size="0.1" mass="1.0" rgba="{color}"
            solref="{solref}" friction="1.0 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>
"""

friction_cfgs = [
    {"label": "Low friction", "friction": "0.08 0.001 0.0001", "color": "0.2 0.7 0.3 1"},
    {"label": "Medium friction", "friction": "0.4 0.003 0.0001", "color": "0.2 0.5 0.8 1"},
    {"label": "High friction", "friction": "1.2 0.01 0.0005", "color": "0.8 0.4 0.2 1"},
]

softness_cfgs = [
    {"label": "Stiff contact", "solref": "-8000 -120", "color": "0.2 0.5 0.8 1"},
    {"label": "Soft contact", "solref": "-250 -12", "color": "0.8 0.4 0.2 1"},
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: sliding distance under different friction values
ax = axes[0]
for cfg in friction_cfgs:
    m = mujoco.MjModel.from_xml_string(SLIDE_XML.format(**cfg))
    d = mujoco.MjData(m)
    d.qvel[0] = 1.8

    times, xs = [], []
    for _ in range(1200):
        mujoco.mj_step(m, d)
        times.append(d.time)
        xs.append(d.qpos[0])

    ax.plot(times, xs, label=cfg["label"], linewidth=1.8)

ax.set_title("Tangential effect: sliding distance depends on friction")
ax.set_xlabel("time (s)")
ax.set_ylabel("x position (m)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# Right: rebound / settling under different contact softness values
ax = axes[1]
for cfg in softness_cfgs:
    m = mujoco.MjModel.from_xml_string(DROP_XML.format(**cfg))
    d = mujoco.MjData(m)

    times, heights = [], []
    for _ in range(1200):
        mujoco.mj_step(m, d)
        times.append(d.time)
        heights.append(d.qpos[2])

    ax.plot(times, heights, label=cfg["label"], linewidth=1.8)

ax.set_title("Normal effect: contact softness changes impact behaviour")
ax.set_xlabel("time (s)")
ax.set_ylabel("ball height (m)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

fig.suptitle("Friction and compliance are different contact effects", fontsize=12)
fig.tight_layout()
plt.show()


---
# 5. Rendering and Synthetic Data <a id="rendering"></a>

For **state-based** policies (joint angles, velocities), rendering quality doesn't matter.  But for **vision-based** policies — or for training perception models — the renderer becomes critical.

## Rasterisation vs Ray Tracing

| | Rasterisation | Ray / Path Tracing |
|-|--------------|-------------------|
| **How** | Project triangles onto screen, shade per-fragment | Trace light rays from camera through scene, bounce | 
| **Speed** | Real-time (1–10 ms/frame) | Slow (100 ms–seconds; RTX hardware helps) |
| **Lighting** | Approximated (shadow maps, SSAO) | Physically accurate (global illumination, caustics) |
| **Reflections** | Screen-space or cube maps | Exact recursive reflections |
| **Sim-real gap** | Larger — lighting artifacts, flat shadows | Smaller — closer to real camera images |
| **Used by** | MuJoCo, PyBullet, Gazebo | Isaac Sim (RTX), Blender (Cycles) |

> For **control** tasks with state observations, rasterisation is perfectly fine (and much faster).  For **vision-based sim-to-real**, investing in better rendering (or compensating with domain randomization) is essential.

## Synthetic data pipeline

A typical pipeline for training perception from simulation:

```
Scene setup (USD/MJCF)
  │
  ├─► Randomize: object poses, textures, lighting, camera
  │
  ├─► Render: RGB + Depth + Segmentation masks + Pose labels
  │
  ├─► (Optional) Post-process: motion blur, noise, lens distortion
  │
  └─► Train: object detection, pose estimation, segmentation
```

**Key enablers:**
- **Domain randomisation** on visual parameters bridges the gap even with rasterised rendering (more in Section 7).
- **Structured light / depth** in simulation is noise-free — adding realistic noise models (e.g. Intel RealSense noise) is important.
- **Isaac Sim Replicator** and **BlenderProc** are popular tools for procedural synthetic data generation.

> **Key takeaway:** Rendering quality is a dial you can turn.  Higher fidelity reduces the visual reality gap but costs more compute.  Domain randomization is the cheaper alternative that often works surprisingly well.


## What matters so far?

At this point we have four separate design choices:

- **Description format** decides how expressive the robot/world model is.
- **Platform** decides the surrounding ecosystem and throughput.
- **Physics engine** decides which contact and constraint effects are believable.
- **Renderer** decides whether the simulator is useful for control only or also
  for perception and synthetic data.

The next question is the real one: even with all of those choices made well,
why does a policy still fail on hardware?


---
# 6. The Reality Gap <a id="reality-gap"></a>

## Part II. Bridging the Reality Gap

A policy trained in simulation encounters three kinds of mismatch when deployed on a real robot:

| Gap type | What differs | Examples |
|----------|-------------|---------|
| **Visual** | Appearance, lighting, textures, sensor noise | Sim has perfect textures; real camera has motion blur, glare, different background |
| **Dynamics** | Physics parameters, contact behaviour | Mass estimation is off by 15 %; real friction varies across table surface; cable catches on edge |
| **Action** | Actuator response, latency, calibration | Motor saturates earlier than modelled; 5 ms communication delay not in sim; joint zero offset |

### Why small gaps compound

Even a 5 % error in predicted next-state can compound over a 100-step trajectory into a completely different outcome.  Contact-rich tasks (grasping, insertion) are especially sensitive because small force/position errors push the object into a qualitatively different contact mode.

### Illustration: same PD controller, sim vs real

Consider a simple reaching task.  In simulation the arm converges smoothly.  On the real robot, unmodelled joint friction and a 10 ms delay cause overshoot and oscillation:

```
Sim:   ────────────────────────► goal  (smooth convergence)

Real:  ───╱╲──╱╲──╱╲──────────► goal  (overshoot from delay + friction)
```

The techniques in the next four sections all aim to bridge this gap — each from a different angle.

> **Key takeaway:** The reality gap is not one problem but three (visual, dynamics, action).  The best sim-to-real strategies address all three, usually by combining multiple techniques.


---
# 7. Domain Randomization <a id="domain-randomization"></a>

**Core idea:** instead of making the simulator *match* reality exactly, make the policy *robust* to a wide range of simulator settings.  If the training distribution is broad enough, reality becomes just another sample.

## What to randomize

| Category | Parameters | Typical range |
|----------|-----------|---------------|
| **Visual** | Textures, colours, lighting direction/intensity, camera pose, FOV, background | Random textures; light intensity ×0.5–2.0; camera position ±5 cm |
| **Dynamics** | Mass, friction, damping, joint stiffness, gravity | ×0.5–2.0 of nominal values |
| **Action** | Gaussian noise on torques, random delays, control gain perturbation | σ = 5–20 % of action range; 1–5 step delay |

## Case study 1: OpenAI — Solving Rubik's Cube with a Robot Hand (2019)

OpenAI's Dactyl system trained a dexterous manipulation policy **entirely in simulation** using massive DR:

- **Dynamics DR**: randomised cube size (±10 %), mass (×0.5–5.0), all friction coefficients, joint damping, actuator gains.
- **Visual DR**: randomised lighting, camera position, table textures, cube colours.
- **Observation noise**: added to all proprioceptive and visual inputs.
- **Action perturbations**: latency, dropped commands, noisy torques.

The result: **zero-shot transfer** to a physical Shadow Hand solving a Rubik's cube — despite never seeing the real robot during training.

Key insight: the *volume* of randomization matters.  Training across ~100 randomization dimensions forced the policy to rely only on features that are invariant across all variations — which happen to be the features that also work in reality.

## Case study 2: VIRAL — Humanoid Loco-Manipulation (2024)

VIRAL (Visual Randomization for humanoid whole-body control) demonstrates large-scale visual DR for zero-shot deployment:

- **Visual DR over**: lighting conditions, material properties, camera intrinsics, image quality degradation (blur, noise, compression).
- **Dynamics alignment**: system identification of hand kinematics and camera extrinsics, then randomize within identified bounds.
- **Result**: a humanoid that manipulates objects while walking, transferred zero-shot from Isaac Sim to hardware.

The key contribution: *combining* visual DR with careful system identification — DR alone wasn't enough; SysID alone wasn't robust enough.  Together they achieved reliable transfer.

## Interactive demo: DR on our arm model

We first render the **nominal** MuJoCo scene, then compare it against several
randomized variants.  This is closer to the actual training setup: one nominal
model plus a distribution of perturbations around it.


In [ ]:
from lib.domain_randomization import DRConfig, render_dr_grid
from lib.viz import show_image_grid

xml_source = open("assets/demo_arm.mjcf").read()

cfg = DRConfig(
    mass_scale=(0.5, 2.0),
    friction_scale=(0.3, 3.0),
    damping_scale=(0.5, 2.0),
    geom_rgba_noise=0.25,
    light_pos_noise=1.0,
    light_diffuse_noise=0.4,
    enabled={"mass", "friction", "damping", "rgba", "light"},
)

frames = render_dr_grid(
    xml_source,
    cfg,
    n_variants=5,
    seed=42,
    width=360,
    height=270,
    include_nominal=True,
)

titles = ["Nominal"] + [f"Randomized sample {i}" for i in range(1, len(frames))]
show_image_grid(frames, titles=titles, cols=3)
plt.suptitle("Domain Randomization: nominal scene plus randomized variants", fontsize=12)
plt.show()


---
# 8. Domain Adaptation and System Identification <a id="adaptation-sysid"></a>

DR makes the policy robust by training on a *broad* distribution.  The two techniques in this section try to *narrow* the gap instead.

## Domain Adaptation (DA)

**Goal:** transform simulator observations so they *look like* real observations (or vice versa), without changing the dynamics.

### Paired adaptation
If you can collect matching sim-real image pairs (same scene, same viewpoint), train a supervised image-to-image model (e.g. pix2pix) to translate sim → real.

### Unpaired adaptation (CycleGAN)
Usually you *can't* collect exact pairs.  CycleGAN learns two generators $G_{\text{sim→real}}$ and $G_{\text{real→sim}}$ with a cycle-consistency loss:

$$\mathcal{L}_{\text{cycle}} = \| G_{\text{real→sim}}(G_{\text{sim→real}}(x_{\text{sim}})) - x_{\text{sim}} \|_1
+ \| G_{\text{sim→real}}(G_{\text{real→sim}}(x_{\text{real}})) - x_{\text{real}} \|_1$$

This preserves semantic content (object positions, shapes) while transferring visual style.

**Workflow:**
1. Collect unpaired images from sim and real.
2. Train CycleGAN.
3. During RL training, pass sim images through $G_{\text{sim→real}}$ before feeding to the policy — the policy sees "real-looking" images.

**Limitation:** CycleGAN can hallucinate or remove objects.  Careful validation is needed.

## System Identification (SysID)

**Goal:** find physical parameters $\theta^*$ (mass, friction, damping, delays) so that the simulator matches real-robot trajectories.

$$\theta^* = \arg\min_\theta \sum_i \| \tau_{\text{sim}}(\theta) - \tau_{\text{real}} \|^2$$

where $\tau$ are state trajectories under the same actions.

**Practical workflow:**
1. Execute a set of "exciting" motions on the real robot, record states.
2. Replay the same actions in sim with candidate parameters.
3. Optimise (gradient-free: CMA-ES, Bayesian opt; or gradient-based if sim is differentiable).

### SysID + DR (the common combo)

SysID gives a point estimate $\hat{\theta}$ with some uncertainty.  DR then samples around $\hat{\theta}$:

$$\theta \sim \mathcal{U}[\hat{\theta} - \delta, \; \hat{\theta} + \delta]$$

This is exactly the approach VIRAL uses: identify the robot-specific parameters, then randomize within the identified confidence region.

> **Key takeaway:** DA addresses the *visual* gap; SysID addresses the *dynamics* gap.  Both are complementary to DR and are often combined with it.


---
# 9. Teacher–Student Architectures <a id="teacher-student"></a>

A recurring problem: the observations available **in simulation** (ground-truth object pose, contact forces, terrain map) are richer than what the **real robot** can see (noisy camera, proprioception only).

**Teacher–Student distillation** solves this in two phases:

## Phase 1: Train a Teacher with privileged information

In simulation we have access to the full state.  The teacher policy $\pi_T$ receives **privileged observations** $o_{\text{priv}}$:

- Ground-truth object pose and velocity
- Contact forces and normals
- Terrain heightmap under the robot's feet
- Exact physical parameters (mass, friction)

Because the teacher sees everything, it learns a better policy faster — the RL problem is much easier with full state.

## Phase 2: Distill into a Student with deployable observations

The student policy $\pi_S$ receives only **deployable observations** $o_{\text{deploy}}$:

- Camera images (RGB / depth)
- Proprioception (joint angles, velocities, torques)
- IMU readings

The student is trained via **supervised learning** (behavioural cloning) on the teacher's behaviour:

$$\mathcal{L}_{\text{distill}} = \mathbb{E}_{o \sim \mathcal{D}} \left[ \| \pi_S(o_{\text{deploy}}) - \pi_T(o_{\text{priv}}) \|^2 \right]$$

where $\mathcal{D}$ is a dataset of rollouts from the teacher policy.

## Architecture diagram

```
┌──────────────────────────────┐
│  SIMULATION (training)       │
│                              │
│  Privileged state ──► Teacher policy ──► actions
│       │                       │
│       │  rollout dataset      │
│       ▼                       │
│  Deployable obs ──► Student policy ──► actions (distillation)
│                              │
└──────────────────────────────┘
                │
                ▼
┌──────────────────────────────┐
│  REAL ROBOT (deployment)     │
│                              │
│  Camera + proprio ──► Student policy ──► actions
│                              │
└──────────────────────────────┘
```

## Examples in practice

| System | Teacher obs | Student obs | Task |
|--------|-----------|-------------|------|
| **ANYmal** (ETH) | Terrain heightmap, foot contacts | Proprioception + history | Legged locomotion over rough terrain |
| **OpenAI Rubik's** | Cube pose + finger contacts | Camera + fingertip sensors | Dexterous manipulation |
| **DexMV** | Object 6-DoF pose | RGB images | Vision-based dexterous grasping |

> **Key takeaway:** Teacher-student is the standard recipe for deploying RL policies with limited real-world sensing.  The teacher makes RL tractable; the student makes deployment possible.


---
# 10. Digital Twins <a id="digital-twins"></a>

A **digital twin** is a high-fidelity, continuously-updated simulation model of a *specific* physical system — not a generic robot model, but *your* robot in *your* environment.

## From generic sim to digital twin

| Aspect | Generic simulator | Digital twin |
|--------|------------------|-------------|
| **Calibration** | Nominal parameters | Identified from real-robot data |
| **Environment** | Abstract arena | 3D scan of actual workspace |
| **Synchronisation** | One-time setup | Continuous update from telemetry |
| **Fidelity** | "Close enough" | Validated against real trajectories |

## Building a digital twin

1. **Geometry**: 3D scan the workspace (LiDAR, photogrammetry, or depth cameras — recall Week 5).
2. **Physics calibration**: System identification of robot + object parameters.
3. **Sensor models**: Calibrate camera intrinsics/extrinsics, noise models for depth, IMU bias.
4. **Continuous sync**: Stream joint states, sensor data, task outcomes back into the twin.  Detect when the twin diverges (model drift) and trigger re-calibration.

## Use cases

- **Predictive maintenance**: Simulate wear, detect anomalies before they cause failure.
- **Online planning**: Plan the next manipulation sequence in the twin, then execute on the real robot.  If the twin prediction diverges from reality mid-execution, replan.
- **Fleet management**: A factory with 50 identical robots can have 50 individualised digital twins, each tracking its own wear and calibration state.
- **Sim-to-real refinement**: Use the digital twin as a fine-tuning environment after initial training with DR.

## Limitations

- **Maintenance cost**: Keeping the twin synchronised requires infrastructure (data pipelines, monitoring, re-calibration triggers).
- **Fidelity ceiling**: Some phenomena (deformable objects, liquids, thermal effects) are still hard to model accurately even in a calibrated twin.
- **Diminishing returns**: For many RL tasks, DR + SysID is "good enough" and a full digital twin is overkill.

> **Key takeaway:** A digital twin is the gold standard for sim-to-real fidelity, but it comes with maintenance overhead.  Reserve it for high-value, safety-critical, or fleet-scale deployments.


## What have we gained from Part II?

We now have a menu of transfer tools with different data requirements:

- **Domain randomization**: cheap if simulation is available, but broad and indirect.
- **Domain adaptation**: useful for vision, but requires image data and careful validation.
- **System identification**: needs real logs, but gives task-relevant physical calibration.
- **Teacher-student distillation**: uses privileged simulation signals to make deployment feasible.
- **Digital twins**: highest fidelity, highest maintenance cost.

The remaining question is how to actually **use** these simulators and transfer
tools to learn policies: reward design, imitation, dataset aggregation, and
human correction.


---
# 11. RL Formulation for Robotics <a id="rl-formulation"></a>

## Part III. Learning from Interaction

Reinforcement learning gives a robot a way to discover control strategies by **trial and error**.  Formalising the problem correctly is half the battle.

## MDP for a robotic task

| MDP component | Robotics instantiation | Example (reaching) |
|--------------|----------------------|-------------------|
| **State** $s$ | Robot config + environment | Joint angles, velocities, object pose |
| **Action** $a$ | Motor commands | Joint torques or target velocities |
| **Transition** $P(s' \mid s, a)$ | Physics (simulator or real) | MuJoCo `mj_step` |
| **Reward** $r(s, a)$ | Task-specific signal | $-\|x_{\text{ee}} - x_{\text{goal}}\|$ |
| **Discount** $\gamma$ | How much to value future | 0.99 (long horizon) |

The agent maximises the **expected discounted return**:

$$J(\pi) = \mathbb{E}_{\pi} \left[ \sum_{t=0}^{T} \gamma^t \, r(s_t, a_t) \right]$$

## Reward design

Getting the reward right is arguably the hardest part of RL in robotics.

### Dense vs Sparse

| | Dense reward | Sparse reward |
|-|-------------|---------------|
| **Signal** | Every step (e.g. distance to goal) | Only at success (e.g. +1 when grasped) |
| **Learning** | Faster — continuous gradient signal | Slower — must discover success by exploration |
| **Risk** | Reward hacking, local optima | Exploration bottleneck |
| **Example** | $r = -\|x_{ee} - x_{goal}\|^2 - 0.01 \|a\|^2$ | $r = \mathbb{1}[\text{object placed}]$ |

### Reward shaping pitfalls

> **Goodhart's law for robots:** once a metric becomes a target, it ceases to be a good metric.

Common failure modes:
- **Reward hacking**: Robot finds a loophole.  "Minimise distance to target" → arm vibrates near target without actually reaching.
- **Competing terms**: $r = r_{\text{reach}} + r_{\text{grasp}} + r_{\text{lift}} + r_{\text{place}}$ — weighting these is fragile.
- **Shaping bias**: Adding intermediate rewards (e.g. "approach object") can prevent discovering better strategies.

**Best practices:**
1. Start with the *simplest possible* reward (sparse success + small action penalty).
2. Add dense terms only if exploration fails.
3. Validate that the learned behaviour actually solves the task, not just maximises the proxy.

Let's visualise dense vs sparse reward landscapes for a 2D reaching task:


In [ ]:
from lib.viz import plot_reward_landscape_2d

goal = np.array([0.5, 0.3])

def dense_reward(x, y):
    return -np.sqrt((x - goal[0])**2 + (y - goal[1])**2)

def sparse_reward(x, y):
    dist = np.sqrt((x - goal[0])**2 + (y - goal[1])**2)
    return 1.0 if dist < 0.15 else 0.0

def shaped_reward(x, y):
    """Dense + bonus near goal — can create misleading gradients."""
    dist = np.sqrt((x - goal[0])**2 + (y - goal[1])**2)
    return -dist + 2.0 * np.exp(-dist**2 / 0.02)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (fn, title) in zip(axes, [
    (dense_reward, "Dense: $r = -\\|x - x_g\\|$"),
    (sparse_reward, "Sparse: $r = \\mathbb{1}[\\|x - x_g\\| < 0.15]$"),
    (shaped_reward, "Shaped: dense + Gaussian bonus"),
]):
    xs = np.linspace(-1, 2, 200)
    ys = np.linspace(-1, 1.5, 200)
    X, Y = np.meshgrid(xs, ys)
    Z = np.vectorize(fn)(X, Y)
    cf = ax.contourf(X, Y, Z, levels=40, cmap="viridis")
    fig.colorbar(cf, ax=ax, shrink=0.8)
    ax.scatter(*goal, marker="*", s=200, c="red", edgecolors="k", zorder=5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
fig.suptitle("Reward landscape comparison (goal = red star)", fontsize=12)
fig.tight_layout()
plt.show()


---
# 12. Imitation Learning and DAgger <a id="imitation-dagger"></a>

Sometimes we don't have a reward function — but we *do* have an expert (human teleoperator, motion planner, or scripted policy).  **Imitation learning** extracts a policy from expert demonstrations.

## Behavioral Cloning (BC)

The simplest approach: treat it as supervised learning.

$$\pi_{\text{BC}} = \arg\min_\pi \; \mathbb{E}_{(s, a) \sim \mathcal{D}_{\text{expert}}} \left[ \| \pi(s) - a \|^2 \right]$$

**Problem — covariate shift:**  At training time, states come from the expert's distribution $d_{\text{expert}}$.  At test time, the learned policy makes small errors, visits slightly different states $d_{\pi}$, and those errors **compound**:

$$\text{BC error} \leq T \cdot \epsilon_{\text{policy}} + T^2 \cdot \epsilon_{\text{drift}}$$

After a few time steps the policy is in territory it never saw during training, and performance degrades rapidly.  This is the fundamental limitation of BC.

## DAgger: Dataset Aggregation

**DAgger** (Ross et al., 2011) fixes covariate shift by *aggregating* data from the learner's own state distribution:

---

**Algorithm: DAgger**

1. Collect initial expert dataset $\mathcal{D}_0 = \{(s, a^*)\}$
2. **For** round $i = 1, \ldots, N$:
   1. Train policy $\hat{\pi}_i$ on $\mathcal{D}_{0:i-1}$
   2. **Roll out** $\hat{\pi}_i$ in the environment → collect visited states $\{s_t\}$
   3. **Query expert** for actions at these states: $a^*_t = \pi^*(s_t)$
   4. **Aggregate**: $\mathcal{D}_i = \mathcal{D}_{i-1} \cup \{(s_t, a^*_t)\}$
3. **Return** $\hat{\pi}_N$

---

**Why it works:** each round, the dataset expands toward the learner's own
state distribution rather than staying frozen at the expert's demonstration
distribution.  In the original analysis this is expressed through a
**no-regret online learning** argument: if the supervised learner keeps making
low regret updates on the aggregated dataset, the deployed policy can approach
expert-level performance much more closely than plain BC.

**Cost:** requires an interactive expert that can label new states — not always available.

## DAgger Variants

| Variant | Key modification | When to use |
|---------|-----------------|-------------|
| **HG-DAgger** (Human-Gated) | Human only intervenes and labels when the policy enters "dangerous" states | Reduces expert labelling burden; good when expert time is expensive |
| **CR-DAgger** (Compliant Residual) | Human provides small residual corrections through a compliant interface while the base policy runs | Contact-rich tasks where full expert demonstration is hard; e.g. insertion, polishing |

### CR-DAgger in detail

In **CR-DAgger**, the executed action is:

$$a_{\text{exec}} = \hat{\pi}(s) + \Delta a_{\text{human}}$$

The human holds a compliant handle and pushes/pulls to correct the robot.  The system records $(s, a_{\text{exec}})$ and retrains the policy to incorporate the corrections.  This is especially valuable for tasks where:
- The expert can't easily demonstrate from scratch (e.g. tight peg insertion).
- Small corrections in force/position are more intuitive than full teleoperation.

## Human-in-the-Loop RL

A broader framework that combines:
- **Initial demonstrations** → BC warm-start.
- **Corrections** → DAgger-style dataset expansion.
- **Autonomous exploration** → RL with reward (possibly shaped by human preferences).
- **Safety filters** → Prevent dangerous states during exploration.

This continuum spans pure IL to pure RL, with most practical systems sitting somewhere in between.

## When to use what

| Approach | Good when | Watch out for |
|----------|----------|---------------|
| **BC** | Good demos available, short horizons, safety-critical initial deployment | Covariate shift on long horizons |
| **DAgger** | Interactive expert available, moderate horizons | Expert labelling cost |
| **RL** | Clear reward signal, simulator available, long-horizon optimisation | Reward engineering, sample efficiency |
| **IL + RL hybrid** | IL for warm-start, RL for refinement and robustness | Complexity, training stability |
| **IL + DAgger + DR** | Real-robot manipulation — the most common recipe today | Infrastructure overhead |

## Demo: BC vs DAgger on a 2D navigation task

The expert navigates toward $y = 0$ while moving forward, compensating for a nonlinear drag.  BC is trained on 300 expert samples (all near $|y| < 0.3$).  DAgger runs 12 rounds of data aggregation.

To make the lesson honest, DAgger is not only **evaluated** on an
off-distribution start at $y_0 = 1.5$; during aggregation it is also
**reset into that harder region** and queries the expert there.  Observe how
this changes the policy relative to BC, which only ever sees the original
expert dataset:


In [ ]:
from lib.imitation import behavioral_cloning, dagger, make_2d_nav_env

rng = np.random.default_rng(42)

# Training environment: seeded for reproducibility
train_step_fn, expert_fn, gen_data = make_2d_nav_env(noise_std=0.02, seed=1)
expert_states, expert_actions = gen_data(n=300, rng=rng)

# --- Behavioral Cloning ---
bc_policy, bc_losses = behavioral_cloning(
    expert_states,
    expert_actions,
    epochs=500,
    lr=3e-3,
    rng=rng,
)

# --- DAgger ---
# Important: during data aggregation we deliberately reset into the harder
# off-distribution region, so the demo matches the narrative.
off_dist_sampler = lambda local_rng: np.array([0.0, local_rng.uniform(0.8, 1.6)])

dagger_policy, dagger_losses, _ = dagger(
    env_step_fn=train_step_fn,
    expert_fn=expert_fn,
    initial_states=expert_states,
    initial_actions=expert_actions,
    n_rounds=12,
    rollout_len=30,
    n_rollouts=12,
    train_epochs=300,
    lr=3e-3,
    rng=rng,
    reset_sampler=off_dist_sampler,
)


def rollout(policy_fn, step_fn, s0, horizon=150):
    states = []
    s = np.array(s0, dtype=float)
    for _ in range(horizon):
        a = policy_fn(s)
        s = step_fn(s, a)
        states.append(s.copy())
    return np.array(states)


start_state = np.array([0.0, 1.5])

# Evaluation environments: same dynamics, same noise seed, separate closures
bc_eval_step, _, _ = make_2d_nav_env(noise_std=0.02, seed=10)
dagger_eval_step, _, _ = make_2d_nav_env(noise_std=0.02, seed=10)
expert_eval_step, _, _ = make_2d_nav_env(noise_std=0.02, seed=10)

bc_rollout = rollout(lambda s: bc_policy.predict(s[np.newaxis])[0], bc_eval_step, start_state)
dagger_rollout = rollout(lambda s: dagger_policy.predict(s[np.newaxis])[0], dagger_eval_step, start_state)
expert_rollout = rollout(expert_fn, expert_eval_step, start_state)

# --- Visualise ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: trajectories in (x, y) space
ax = axes[0]
ax.plot(expert_rollout[:, 0], expert_rollout[:, 1], label="Expert", c="tab:green", linewidth=2)
ax.plot(bc_rollout[:, 0], bc_rollout[:, 1], label="BC", c="tab:red", linewidth=1.5, linestyle="--")
ax.plot(dagger_rollout[:, 0], dagger_rollout[:, 1], label="DAgger", c="tab:blue", linewidth=1.5)
ax.axhline(0, color="gray", linestyle=":", alpha=0.5, label="target (y=0)")
ax.scatter([0], [1.5], marker="o", s=80, c="k", zorder=5, label="evaluation start")
ax.fill_between([-1, 20], -0.3, 0.3, alpha=0.1, color="green", label="BC demo region")
ax.fill_between([-1, 20], 0.8, 1.6, alpha=0.08, color="tab:blue", label="DAgger reset region")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("DAgger adds labels from states BC never sees")
ax.legend(fontsize=8)
ax.set_xlim(-0.5, max(bc_rollout[:, 0].max(), dagger_rollout[:, 0].max()) + 1)

# Right: y deviation over time
ax = axes[1]
ax.plot(expert_rollout[:, 1], label="Expert", c="tab:green", linewidth=2)
ax.plot(bc_rollout[:, 1], label="BC", c="tab:red", linewidth=1.5, linestyle="--")
ax.plot(dagger_rollout[:, 1], label="DAgger", c="tab:blue", linewidth=1.5)
ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax.axhspan(-0.3, 0.3, alpha=0.1, color="green")
ax.set_xlabel("Timestep")
ax.set_ylabel("y (deviation from target)")
ax.set_title("Off-distribution evaluation with matched stochasticity")
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()


## Closing synthesis: which tool fixes which problem?

| Technique | Mainly targets | Requires | Typical failure mode |
|-----------|----------------|----------|----------------------|
| **Domain randomization** | Broad visual / dynamics / action mismatch | A configurable simulator | Randomization range misses the real robot |
| **Domain adaptation** | Visual gap | Sim + real images | Translation changes semantics or hides errors |
| **System identification** | Dynamics gap | Real trajectories under known actions | Overfits nominal conditions, misses variability |
| **Teacher-student** | Privileged-state vs deployable-observation gap | Strong simulator + distillation dataset | Student loses critical information available to teacher |
| **Digital twin** | Robot-specific fidelity gap | Calibration pipeline + telemetry | Too costly to maintain relative to task value |
| **Behavioral cloning** | Fast policy initialization | Demonstrations | Covariate shift outside expert distribution |
| **DAgger / HITL corrections** | Covariate shift during deployment | Interactive expert / human oversight | Labelling burden and safety constraints |
| **RL** | Long-horizon performance refinement | Reward + simulator + compute | Reward hacking, poor sample efficiency |

> **Default modern recipe:** identify a reasonable simulator, randomize around
> the uncertain parameters, warm-start from demonstrations if available, and use
> DAgger or RL only where the task genuinely needs it.
